## Test 02: one configurable pair and a reusable rule-testing pipeline

In [ ]:
PAIR = ("USD", "UBT")
SIGNAL_PAIR = PAIR
RULE_SET = "bollinger"
CSV_PATH = "data/etf_daily_joint.csv"
START_DATE = None
END_DATE = None

BASELINES = {
    "bollinger": {"length": 20, "ma_type": "SMA", "stdev": 2.0},
    "rsi": {"rsi_window": 3, "entry_low": 10, "entry_high": 90, "exit_risk": 60, "exit_bond": 40},
}
GRIDS = {
    "bollinger": {
        "length": [5, 10, 15, 20, 30, 40, 60],
        "ma_type": ["SMA", "EMA", "WMA"],
        "stdev": [1.0, 1.5, 2.0, 2.5, 3.0],
    },
    "rsi": {
        "rsi_window": [2, 3, 4, 5, 7, 10, 14],
        "entry_low": [5, 10, 15, 20, 25, 30],
        "entry_high": [70, 75, 80, 85, 90, 95],
        "exit_risk": [50, 60, 70, 80],
        "exit_bond": [20, 30, 40, 50],
    },
}
BASELINE_PARAMS = BASELINES[RULE_SET]
PARAMETER_GRID = GRIDS[RULE_SET]


## Load the selected pair; closing signals execute at the next observed session open

In [ ]:
from pathlib import Path
from itertools import product
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display

risk, bond = PAIR
assert risk != bond and len(SIGNAL_PAIR) == 2
source_path = Path(CSV_PATH)
prices = pd.read_csv(source_path, index_col="date", parse_dates=["date"]).sort_index()
assert prices.index.is_unique and prices.index.is_monotonic_increasing
if START_DATE is not None:
    prices = prices.loc[pd.Timestamp(START_DATE):]
if END_DATE is not None:
    prices = prices.loc[:pd.Timestamp(END_DATE)]
required = list(dict.fromkeys(
    [f"{ticker}_{field}" for ticker in PAIR for field in ["open", "close", "adj_close"]]
    + [f"{ticker}_adj_close" for ticker in SIGNAL_PAIR]
))
if len(prices) < 30 or not set(required).issubset(prices.columns):
    raise ValueError("Run the data notebook first; at least 30 sessions and all selected tickers are required")
if not np.isfinite(prices[required].to_numpy()).all() or (prices[required] <= 0).any().any():
    raise ValueError("Selected prices must be finite, positive and complete; no filling is applied")
output_dir = Path("data") / f"test_02_{risk}_{bond}" / RULE_SET
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Trade: {risk}/{bond}; signal: {SIGNAL_PAIR[0]}/{SIGNAL_PAIR[1]}; rules: {RULE_SET}")
print(f"Sample: {prices.index[0].date()} to {prices.index[-1].date()}; {len(prices):,} sessions")
print("Fully invested in one ETF or cash; before costs; cash earns 0%; no forced final liquidation")


## Features: select Bollinger Bands or RSI in the first cell; Bollinger width uses rolling population standard deviation (ddof=0)

In [ ]:
def wilder_rsi(series, window):
    window = int(window)
    if window < 1 or window >= len(series):
        raise ValueError("RSI window must be positive and shorter than the sample")
    change = series.diff()
    gains = change.clip(lower=0)
    losses = -change.clip(upper=0)
    avg_gain = gains.iloc[1:window + 1].mean()
    avg_loss = losses.iloc[1:window + 1].mean()
    values = pd.Series(np.nan, index=series.index)
    for i in range(window, len(series)):
        if i > window:
            avg_gain = ((window - 1) * avg_gain + gains.iloc[i]) / window
            avg_loss = ((window - 1) * avg_loss + losses.iloc[i]) / window
        values.iloc[i] = (50 if avg_gain == 0 and avg_loss == 0 else
                          100 if avg_loss == 0 else 100 - 100 / (1 + avg_gain / avg_loss))
    return values

def valid_parameters(parameters):
    if RULE_SET == "bollinger":
        return (
            parameters.length.ge(2) & parameters.length.lt(len(prices))
            & parameters.length.eq(parameters.length.astype(int))
            & parameters.ma_type.isin(["SMA", "EMA", "WMA"])
            & parameters.stdev.gt(0) & np.isfinite(parameters.stdev)
        )
    if RULE_SET == "rsi":
        return (
            parameters.rsi_window.ge(1) & parameters.rsi_window.lt(len(prices))
            & parameters.rsi_window.eq(parameters.rsi_window.astype(int))
            & parameters.entry_low.ge(0) & parameters.entry_high.le(100)
            & parameters.entry_low.lt(parameters.exit_bond)
            & parameters.exit_bond.le(parameters.exit_risk)
            & parameters.exit_risk.lt(parameters.entry_high)
        )
    raise ValueError("RULE_SET must be bollinger or rsi")

def moving_average(series, length, ma_type):
    if ma_type == "SMA":
        return series.rolling(length, min_periods=length).mean()
    if ma_type == "EMA":
        return series.ewm(span=length, adjust=False, min_periods=length).mean()
    if ma_type == "WMA":
        weights = np.arange(1, length + 1, dtype=float)
        return series.rolling(length, min_periods=length).apply(
            lambda values: np.dot(values, weights) / weights.sum(), raw=True
        )
    raise ValueError(f"Unsupported MA type: {ma_type}")

parameter_names = list(PARAMETER_GRID)
parameters = pd.DataFrame(product(*PARAMETER_GRID.values()), columns=parameter_names)
parameters = parameters.loc[valid_parameters(parameters)].reset_index(drop=True)
baseline_parameters = pd.DataFrame([BASELINE_PARAMS])
if parameters.empty or not valid_parameters(baseline_parameters).all():
    raise ValueError("Invalid baseline or empty parameter grid")

features = pd.DataFrame(index=prices.index)
features["spread"] = prices[f"{SIGNAL_PAIR[0]}_adj_close"] / prices[f"{SIGNAL_PAIR[1]}_adj_close"]
all_parameters = pd.concat([parameters, baseline_parameters], ignore_index=True)
if RULE_SET == "bollinger":
    for length in sorted(all_parameters.length.unique()):
        length = int(length)
        features[f"bb_std_{length}"] = features.spread.rolling(length, min_periods=length).std(ddof=0)
    for length, ma_type in all_parameters[["length", "ma_type"]].drop_duplicates().itertuples(index=False, name=None):
        features[f"bb_middle_{int(length)}_{ma_type}"] = moving_average(features.spread, int(length), ma_type)
    length = int(BASELINE_PARAMS["length"])
    middle = features[f"bb_middle_{length}_{BASELINE_PARAMS['ma_type']}"]
    width = BASELINE_PARAMS["stdev"] * features[f"bb_std_{length}"]
    baseline_bands = pd.DataFrame({"spread": features.spread, "middle": middle,
                                   "upper": middle + width, "lower": middle - width})
    ax = baseline_bands.plot(figsize=(14, 5), linewidth=1,
                             title=f"{SIGNAL_PAIR[0]}/{SIGNAL_PAIR[1]} Bollinger Bands: {length}, {BASELINE_PARAMS['ma_type']}, {BASELINE_PARAMS['stdev']} sigma")
    ax.fill_between(features.index, middle - width, middle + width, alpha=0.1)
else:
    for window in sorted(all_parameters.rsi_window.unique()):
        features[f"rsi_{int(window)}"] = wilder_rsi(features.spread, int(window))
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True, constrained_layout=True)
    axes[0].plot(features.spread)
    axes[0].set_title(f"{SIGNAL_PAIR[0]}/{SIGNAL_PAIR[1]} adjusted price spread")
    axes[1].plot(features[f"rsi_{BASELINE_PARAMS['rsi_window']}"])
    for level in [BASELINE_PARAMS[k] for k in ["entry_low", "entry_high", "exit_risk", "exit_bond"]]:
        axes[1].axhline(level, linestyle="--", alpha=0.4)
    axes[1].set_title(f"Baseline Wilder RSI ({BASELINE_PARAMS['rsi_window']})")
plt.show()


## Bollinger rules: below lower → enter risk ETF; above upper → enter bond ETF; above middle → exit risk; below middle → exit bond. RSI remains a separate selectable rule set.

In [ ]:
def make_rules(features, parameters):
    if RULE_SET == "bollinger":
        spread_values = features.spread.to_numpy()[:, None]
        middle = features[[f"bb_middle_{int(row.length)}_{row.ma_type}" for row in parameters.itertuples()]].to_numpy()
        width = features[[f"bb_std_{int(length)}" for length in parameters.length]].to_numpy() * parameters.stdev.to_numpy()
        return {
            "enter_risk": spread_values < middle - width,
            "enter_bond": spread_values > middle + width,
            "exit_risk": spread_values > middle,
            "exit_bond": spread_values < middle,
        }
    values = features[[f"rsi_{int(window)}" for window in parameters.rsi_window]].to_numpy()
    return {
        "enter_risk": values < parameters.entry_low.to_numpy(),
        "enter_bond": values > parameters.entry_high.to_numpy(),
        "exit_risk": values > parameters.exit_risk.to_numpy(),
        "exit_bond": values < parameters.exit_bond.to_numpy(),
    }


## Execution and accounting: next-open fills; exits first, then entries; one position or cash

In [ ]:
market_close = np.column_stack([prices[f"{ticker}_adj_close"] for ticker in PAIR])
market_open = np.column_stack([
    prices[f"{ticker}_open"] * prices[f"{ticker}_adj_close"] / prices[f"{ticker}_close"]
    for ticker in PAIR
])
market_overnight = np.zeros((len(prices), 3))
market_intraday = np.zeros((len(prices), 3))
market_overnight[1:, 1:] = market_open[1:] / market_close[:-1] - 1
market_intraday[:, 1:] = market_close / market_open - 1

def run_backtest(parameters, features, keep_daily=False):
    rules = make_rules(features, parameters)
    count = len(parameters)
    if count == 0 or (keep_daily and count != 1):
        raise ValueError("Daily output requires exactly one configuration")
    for mask in rules.values():
        if mask.shape != (len(prices), count) or mask.dtype != bool:
            raise ValueError("Each rule must be a boolean array with shape (sessions, configurations)")
    if np.any(rules["enter_risk"] & rules["enter_bond"]):
        raise ValueError("Risk and bond entry rules cannot both be true for the same configuration/date")
    holding = np.zeros(count, dtype=int)
    equity = np.ones(count)
    peak = np.ones(count)
    max_dd = np.zeros(count)
    total = np.zeros(count)
    squared = np.zeros(count)
    downside_squared = np.zeros(count)
    entries = np.zeros(count, dtype=int)
    exposed = np.zeros(count, dtype=int)
    rows = []
    for i in range(len(prices)):
        previous = holding.copy()
        if i > 0:
            exit_now = ((holding == 1) & rules["exit_risk"][i - 1]) | ((holding == 2) & rules["exit_bond"][i - 1])
            holding[exit_now] = 0
            flat = holding == 0
            holding[flat & rules["enter_risk"][i - 1]] = 1
            holding[flat & rules["enter_bond"][i - 1]] = 2
        returns = (1 + market_overnight[i, previous]) * (1 + market_intraday[i, holding]) - 1
        total += returns
        squared += returns ** 2
        downside_squared += np.minimum(returns, 0) ** 2
        equity *= 1 + returns
        peak = np.maximum(peak, equity)
        current_dd = equity / peak - 1
        max_dd = np.minimum(max_dd, current_dd)
        entries += (holding != previous) & (holding != 0)
        exposed += (holding != 0) | (previous != 0)
        if keep_daily:
            rows.append((int(previous[0]), int(holding[0]), returns[0], equity[0], current_dd[0]))
    mean = total / len(prices)
    daily_std = np.sqrt(np.maximum((squared - len(prices) * mean ** 2) / (len(prices) - 1), 0))
    volatility = daily_std * np.sqrt(252)
    downside = np.sqrt(downside_squared / len(prices)) * np.sqrt(252)
    sharpe = np.full(count, np.nan)
    sortino = np.full(count, np.nan)
    np.divide(mean * 252, volatility, out=sharpe, where=volatility > 0)
    np.divide(mean * 252, downside, out=sortino, where=downside > 0)
    years = (prices.index[-1] - prices.index[0]).days / 365.25
    summary = parameters.reset_index(drop=True).copy()
    summary["Sharpe"] = sharpe
    summary["Sortino"] = sortino
    summary["CAGR"] = equity ** (1 / years) - 1
    summary["Annualized volatility"] = volatility
    summary["Max drawdown"] = max_dd
    summary["Total return"] = equity - 1
    summary["Exposure"] = exposed / len(prices)
    summary["Entries"] = entries
    daily = None
    if keep_daily:
        daily = pd.DataFrame(rows, index=prices.index,
                             columns=["previous_position", "position", "return", "equity", "drawdown"])
        daily[["previous_position", "position"]] = daily[["previous_position", "position"]].replace({0: "CASH", 1: risk, 2: bond})
        daily["signal_date"] = pd.Series(prices.index, index=prices.index).shift(1)
        changed = daily.position.ne(daily.previous_position)
        assert (daily.loc[changed, "signal_date"] < daily.index[changed]).all()
    return summary, daily


## Baseline assessment: selected rule set and pair, before costs

In [ ]:
percent_columns = ["CAGR", "Annualized volatility", "Max drawdown", "Total return", "Exposure"]
formats = {column: "{:.2%}" for column in percent_columns}
formats.update({"Sharpe": "{:.3f}", "Sortino": "{:.3f}"})
formats.update({column: "{:.0f}" for column in parameter_names + ["Entries"] if column not in ["ma_type", "stdev"]})
if "stdev" in parameter_names:
    formats["stdev"] = "{:.2f}"
baseline_summary, baseline_daily = run_backtest(baseline_parameters, features, keep_daily=True)
display(baseline_summary.style.format(formats))
baseline_trades = baseline_daily.loc[
    baseline_daily.position.ne(baseline_daily.previous_position),
    ["signal_date", "previous_position", "position"]
].rename_axis("execution_date")
display(baseline_trades.head(10))


## In-sample grid search: maximize full-sample Sharpe; no out-of-sample selection

In [ ]:
grid_results, _ = run_backtest(parameters, features)
grid_results = grid_results.sort_values(
    ["Sharpe"] + parameter_names, ascending=[False] + [True] * len(parameter_names),
    na_position="last"
).reset_index(drop=True)
if grid_results.Sharpe.notna().sum() == 0:
    raise ValueError("No configurations have a defined Sharpe ratio")
best_row = grid_results.loc[grid_results.Sharpe.notna()].iloc[0]
best_parameters = grid_results.loc[[best_row.name], parameter_names].reset_index(drop=True).copy()
best_summary, best_daily = run_backtest(best_parameters, features, keep_daily=True)
for metric in ["Sharpe", "CAGR", "Max drawdown", "Total return"]:
    np.testing.assert_allclose(best_summary[metric].iloc[0], best_row[metric], rtol=1e-10, atol=1e-12)
print(f"{len(grid_results):,} configurations; selected by full-sample Sharpe")
display(grid_results.head(10).style.format(formats))
assessment = pd.concat([
    baseline_summary.assign(test="Baseline"),
    best_summary.assign(test="IN-SAMPLE BEST"),
]).set_index("test")
display(assessment.style.format(formats))


## Equity and drawdowns: in-sample best compared with the baseline and both buy-and-hold ETFs

In [ ]:
curves = pd.DataFrame({
    "Baseline": baseline_daily.equity,
    "IN-SAMPLE BEST": best_daily.equity,
    f"{risk} buy and hold": prices[f"{risk}_adj_close"] / prices[f"{risk}_adj_close"].iloc[0],
    f"{bond} buy and hold": prices[f"{bond}_adj_close"] / prices[f"{bond}_adj_close"].iloc[0],
})
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1]}, constrained_layout=True)
curves.plot(ax=axes[0], logy=True, grid=True)
axes[0].set_ylabel("Growth of 1 (log)")
axes[0].set_title(f"{risk}/{bond} {RULE_SET}: IN-SAMPLE BEST selected by full-sample Sharpe; before costs")
for column, line in zip(curves.columns, axes[0].lines):
    dd = curves[column] / curves[column].cummax().clip(lower=1) - 1
    axes[1].plot(curves.index, dd, color=line.get_color(), linewidth=1)
axes[1].set_ylabel("Drawdown")
axes[1].yaxis.set_major_formatter(PercentFormatter(1))
axes[1].grid(alpha=0.3)
fig.savefig(output_dir / "equity_drawdown.png", dpi=140)
plt.show()


## Monthly returns and compounded yearly totals: boundary periods may be partial

In [ ]:
daily_returns = pd.DataFrame({"Baseline": baseline_daily["return"], "IN-SAMPLE BEST": best_daily["return"]})
monthly_returns = (1 + daily_returns).resample("ME").prod(min_count=1) - 1
yearly_returns = (1 + monthly_returns).groupby(monthly_returns.index.year).prod(min_count=1) - 1
limit = max(float(monthly_returns.abs().max().max()), float(yearly_returns.abs().max().max()), 0.001)
fig, axes = plt.subplots(1, 2, figsize=(18, 9), constrained_layout=True)
for name, ax in zip(monthly_returns.columns, axes):
    table = pd.DataFrame({"year": monthly_returns.index.year,
                          "month": monthly_returns.index.month,
                          "return": monthly_returns[name].to_numpy()}).pivot(index="year", columns="month", values="return")
    table = table.reindex(columns=range(1, 13))
    table["Year"] = yearly_returns[name].reindex(table.index)
    values = table.to_numpy()
    image = ax.imshow(np.ma.masked_invalid(values), cmap="RdYlGn", aspect="auto", vmin=-limit, vmax=limit)
    ax.set_title(f"{risk}/{bond} {RULE_SET}: {name}")
    ax.set_xticks(range(13), ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec", "Year"])
    ax.set_yticks(range(len(table)), table.index)
    ax.axvline(11.5, color="black", linewidth=1)
    for row in range(len(table)):
        for col in range(13):
            value = values[row, col]
            if np.isfinite(value):
                ax.text(col, row, f"{value:.1%}", ha="center", va="center", fontsize=7,
                        color="white" if abs(value) > 0.65 * limit else "black")
bar = fig.colorbar(image, ax=axes.tolist(), shrink=0.7)
bar.ax.yaxis.set_major_formatter(PercentFormatter(1))
fig.suptitle("Compounded returns, before costs; best configuration selected in sample")
fig.savefig(output_dir / "monthly_returns.png", dpi=140)
plt.show()


## Return attribution protocol: annual equity and bond contributions

Equity and bond contributions sum exactly to the strategy return. Overnight P&L belongs to the old holding; intraday P&L belongs to the new holding. Period contributions use wealth at each session start relative to wealth at period start. Cash earns 0%; returns are before costs. Boundary years may be partial.

In [ ]:
def attribute_pair(prices, positions, risk, bond, expected_returns):
    if not positions.index.equals(expected_returns.index) or not positions.index.equals(prices.index):
        raise ValueError("Prices, positions and returns must share the same dates")
    if not positions.isin(["CASH", risk, bond]).all():
        raise ValueError("Unknown holding in pair attribution")
    previous = positions.shift(1, fill_value="CASH")
    overnight = pd.DataFrame(0.0, index=prices.index, columns=["Equity", "Bonds"])
    intraday = overnight.copy()
    for asset, sleeve in [(risk, "Equity"), (bond, "Bonds")]:
        close = prices[f"{asset}_adj_close"]
        open_price = prices[f"{asset}_open"] * close / prices[f"{asset}_close"]
        overnight.loc[previous.eq(asset), sleeve] = (open_price / close.shift(1) - 1).loc[previous.eq(asset)]
        intraday.loc[positions.eq(asset), sleeve] = (close / open_price - 1).loc[positions.eq(asset)]
    daily = overnight + intraday.mul(1 + overnight.sum(axis=1), axis=0)
    np.testing.assert_allclose(daily.sum(axis=1), expected_returns, rtol=1e-10, atol=1e-12)
    growth_before_session = (1 + expected_returns).cumprod().shift(1, fill_value=1.0)
    pnl = daily.mul(growth_before_session, axis=0)
    output = {"daily": daily.assign(Total=expected_returns)}
    for label, frequency in [("monthly", "ME"), ("yearly", "YE")]:
        contributions = pnl.resample(frequency).sum(min_count=1).div(
            growth_before_session.resample(frequency).first(), axis=0
        )
        total = (1 + expected_returns).resample(frequency).prod(min_count=1) - 1
        np.testing.assert_allclose(contributions.sum(axis=1), total, rtol=1e-10, atol=1e-12)
        output[label] = contributions.assign(Total=total)
    output["full_sample"] = pnl.sum().to_frame().T
    output["full_sample"]["Total"] = (1 + expected_returns).prod() - 1
    np.testing.assert_allclose(
        output["full_sample"][["Equity", "Bonds"]].sum(axis=1),
        output["full_sample"].Total, rtol=1e-10, atol=1e-12
    )
    return output

def plot_pair_attribution(attribution, risk, bond, title, ax):
    table = attribution["yearly"]
    x = np.arange(len(table))
    equity_values = table.Equity.to_numpy()
    bond_values = table.Bonds.to_numpy()
    bottom = np.where(equity_values * bond_values > 0, equity_values, 0.0)
    ax.bar(x, equity_values, color="tab:blue", label=f"Equity ({risk})")
    ax.bar(x, bond_values, bottom=bottom, color="tab:orange", label=f"Bonds ({bond})")
    ax.scatter(x, table.Total, color="black", marker="D", s=16, label="Strategy return", zorder=3)
    ax.axhline(0, color="gray", linewidth=0.7)
    ax.set_xticks(x, table.index.year, rotation=60)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.set_ylabel("Contribution (percentage points)")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.2)
    ax.legend(fontsize=8)


In [ ]:
pair_attributions = {
    "Baseline": attribute_pair(prices, baseline_daily.position, risk, bond, baseline_daily["return"]),
    "IN-SAMPLE BEST": attribute_pair(prices, best_daily.position, risk, bond, best_daily["return"]),
}
fig, axes = plt.subplots(2, 1, figsize=(14, 10), constrained_layout=True)
for (name, attribution), ax in zip(pair_attributions.items(), axes):
    plot_pair_attribution(attribution, risk, bond, f"{risk}/{bond} {RULE_SET}: {name}", ax)
fig.suptitle("Annual strategy return contributions: equity and bonds; before costs", fontsize=14)
fig.savefig(output_dir / "return_contributions.png", dpi=140)
plt.show()
for period in ["daily", "monthly", "yearly", "full_sample"]:
    table = pd.concat({name: result[period] for name, result in pair_attributions.items()},
                      names=["test", "date"])
    table.to_csv(output_dir / f"return_contributions_{period}.csv")
display(pd.concat({name: result["full_sample"] for name, result in pair_attributions.items()},
                  names=["test", "row"]).droplevel("row").style.format("{:.2%}"))


## Export: pair-specific results and the configuration used for this run

In [ ]:
grid_results.to_csv(output_dir / "grid_results.csv", index=False)
assessment.to_csv(output_dir / "assessment.csv")
baseline_daily.to_csv(output_dir / "baseline_daily.csv", index_label="date")
best_daily.to_csv(output_dir / "in_sample_best_daily.csv", index_label="date")
monthly_returns.to_csv(output_dir / "monthly_returns.csv", index_label="month")
yearly_returns.to_csv(output_dir / "yearly_returns.csv", index_label="year")
manifest = {
    "pair": list(PAIR), "signal_pair": list(SIGNAL_PAIR), "rule_set": RULE_SET,
    "source_csv": str(source_path),
    "source_sha256": hashlib.sha256(source_path.read_bytes()).hexdigest(),
    "sample_start": prices.index[0].date().isoformat(),
    "sample_end": prices.index[-1].date().isoformat(),
    "baseline": BASELINE_PARAMS,
    "parameter_grid": PARAMETER_GRID,
    "configurations_tested": len(grid_results),
    "in_sample_best": json.loads(best_parameters.to_json(orient="records"))[0],
    "selection": "Full-sample maximum annualized Sharpe, zero risk-free rate, 252 sessions/year",
    "execution": "Previous close signal, next observed session open; adjusted price accounting",
    "costs": 0, "cash_return": 0,
    "attribution": "Equity/bond P&L weighted by pre-session wealth relative to period-start wealth; exact additive contributions",
    "rules": ("Enter risk below lower band; enter bond above upper band; exit risk above middle; exit bond below middle"
              if RULE_SET == "bollinger" else "RSI thresholds"),
    "bollinger_std": "Rolling population standard deviation of spread, ddof=0",
    "ma_conventions": "SMA; EMA span=length adjust=False; WMA weights 1..length; all require length observations",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
(output_dir / "run_config.json").write_text(json.dumps(manifest, indent=2) + "\n")
np.testing.assert_allclose((1 + monthly_returns).prod(), (1 + daily_returns).prod(), rtol=1e-12)
print(f"Saved results to {output_dir.resolve()}")
